In [3]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import os
import re
import json
from typing import List, Dict
from litellm import completion

In [4]:
# FUNCION USADA ANTERIORMENTE.... :3
def generate_response(messages: List[Dict]) -> str:
    """Call LLM to get response"""
    response = completion(
        model="groq/openai/gpt-oss-120b",
        messages=messages,
        max_tokens=1024,
        reasoning_effort="low"
    )
    return response.choices[0].message.content

In [5]:
# HERRAMIENTAS TOOLS - ENVIRONMENT INTERFACE
def list_files() -> List[str]:
    """List all files in the current directory."""
    return os.listdir(".")

def read_file(file_name: str) -> str:
    """Read the content of a file."""
    try:
        with open(file_name, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: file '{file_name}' not found."

In [6]:
# EXTRACT MARKDOWN BLOK 
def extract_markdown_block(text: str, block_type: str = "") -> str:
    """Extract content from a markdown code block (e.g. ```action ... ```)."""
    pattern = rf"```{block_type}\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

def parse_action(response: str) -> Dict:
    """Parse the LLM response into a structured action dictionary."""
    try:
        response = extract_markdown_block(response, "action")
        response_json = json.loads(response)
        if "tool_name" in response_json and "args" in response_json:
            return response_json
        else:
            return {"tool_name": "error", "args": {"message": "You must respond with a JSON tool invocation."}}
    except json.JSONDecodeError:
        return {"tool_name": "error", "args": {"message": "Invalid JSON response. You must respond with a JSON tool invocation."}}

In [14]:
# AGENT RULES
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:
- list_files() -> List[str]: List all files in the current directory.
- read_file(file_name: str) -> str: Read the content of a file.
- terminate(message: str): End the agent loop and print a summary to the user.

If a user asks about files, list them before reading.

Every response MUST have an action.
Respond in this format:

```action
{
    "tool_name": "insert tool_name",
    "args": {...fill in any required arguments here...}
}
```

IMPORTANT: All natural-language text you write (such as the "message" argument in the terminate action) must be written in Spanish. Keep JSON keys and tool names in English exactly as shown — only the content/messages should be in Spanish.
"""
}]

In [16]:
# INICIALIZACION DE MEMORIA + LOOP
memory = [
    {"role": "user", "content": "¿Qué archivos hay en este directorio?"}
]

iterations = 0
max_iterations = 10

# The Agent Loop
while iterations < max_iterations:

    prompt = agent_rules + memory

    print("Agent thinking...")
    response = generate_response(prompt)
    print(f"Agent response: {response}")

    action = parse_action(response)

    result = "Action executed"

    if action["tool_name"] == "list_files":
        result = {"result": list_files()}
    elif action["tool_name"] == "read_file":
        result = {"result": read_file(action["args"]["file_name"])}
    elif action["tool_name"] == "error":
        result = {"error": action["args"]["message"]}
    elif action["tool_name"] == "terminate":
        print(action["args"]["message"])
        break
    else:
        result = {"error": "Unknown action: " + action["tool_name"]}

    print(f"Action result: {result}")

    memory.extend([
        {"role": "assistant", "content": response},
        {"role": "user", "content": json.dumps(result)}
    ])

    if action["tool_name"] == "terminate":
        break

    iterations += 1

Agent thinking...
Agent response: ```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action result: {'result': ['01_introduction.ipynb', 'funcionmoduloi.py', '02_buildingAgent.ipynb']}
Agent thinking...
Agent response: ```action
{
    "tool_name": "terminate",
    "args": {
        "message": "En el directorio se encuentran los siguientes archivos: 01_introduction.ipynb, funcionmoduloi.py y 02_buildingAgent.ipynb."
    }
}
```
En el directorio se encuentran los siguientes archivos: 01_introduction.ipynb, funcionmoduloi.py y 02_buildingAgent.ipynb.
